# Self-RAG Verifier — Kaggle

Fine-tune **Flan-T5** to generate Self-RAG reflection tokens and evaluate on ASQA.

## Kaggle settings (before running)

1. **Accelerator:** GPU T4 x2 (or P100)
2. **Internet:** On (downloads `google/flan-t5-base` on first train)
3. **Add Input datasets** (pick one code source + one data source):
   - **Code:** attach a dataset containing this repo *or* rely on git clone below
   - **Data:** attach `labeled_asqa.csv` (e.g. dataset slug `ragas-data`)
4. **Save Version** when finished so `/kaggle/working` outputs persist

Outputs: `models/answer_verifier/`, `results/answer_verifier/metrics_test.json`

## 0. Kaggle bootstrap

In [ ]:
# --- Edit these paths for your Kaggle datasets ---
# How to get code on Kaggle:
#   "git"     -> clone from GitHub into /kaggle/working (default)
#   "dataset" -> use an attached repo zip/dataset
CODE_SOURCE = "git"
GITHUB_REPO = "https://github.com/Htaaxx/ragas-evaluation.git"
CODE_DATASET_PATH = "/kaggle/input/ragas-code/ragas-evaluation"  # if CODE_SOURCE == "dataset"

# Path to labeled_asqa.csv inside your attached data dataset
LABELED_CSV_INPUT = "/kaggle/input/ragas-data/labeled_asqa.csv"

# Training controls
RUN_TRAIN = True
RESUME_TRAINING = False
SPLIT = "test"
CONFIG_PATH = "configs/experiments/rag_verifier.yaml"

# GPU overrides (applied to rag_verifier.yaml in working copy)
GPU_BATCH_SIZE = 8
USE_FP16 = True

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

import torch

if CODE_SOURCE == "git":
    PROJECT_ROOT = Path("/kaggle/working/ragas-evaluation")
    if not PROJECT_ROOT.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", GITHUB_REPO, str(PROJECT_ROOT)],
            check=True,
        )
elif CODE_SOURCE == "dataset":
    src = Path(CODE_DATASET_PATH)
    if not src.exists():
        raise FileNotFoundError(
            f"Code dataset not found at {src}. "
            "Attach your repo dataset or set CODE_SOURCE='git'."
        )
    PROJECT_ROOT = Path("/kaggle/working/ragas-evaluation")
    if PROJECT_ROOT.exists():
        shutil.rmtree(PROJECT_ROOT)
    shutil.copytree(src, PROJECT_ROOT)
else:
    raise ValueError("CODE_SOURCE must be 'git' or 'dataset'")

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(PROJECT_ROOT / "experiments" / "self_rag_verifier"))

print("Project root:", PROJECT_ROOT)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
!pip install -q -e .
!pip install -q sentencepiece matplotlib seaborn

In [ ]:
import yaml
from rag_filtering.config.loader import resolve_path

csv_src = Path(LABELED_CSV_INPUT)
if not csv_src.exists():
    raise FileNotFoundError(
        f"labeled_asqa.csv not found at {csv_src}.\n"
        "Attach a Kaggle dataset containing the file and update LABELED_CSV_INPUT."
    )

csv_dest_dir = resolve_path("data/asqa")
csv_dest_dir.mkdir(parents=True, exist_ok=True)
csv_dest = csv_dest_dir / "labeled_asqa.csv"
shutil.copy(csv_src, csv_dest)
print(f"Copied data -> {csv_dest} ({csv_dest.stat().st_size / 1e6:.1f} MB)")

cfg_path = resolve_path(CONFIG_PATH)
cfg = yaml.safe_load(cfg_path.read_text(encoding="utf-8"))
cfg["training"]["fp16"] = USE_FP16 and torch.cuda.is_available()
cfg["training"]["batch_size"] = GPU_BATCH_SIZE
cfg_path.write_text(yaml.dump(cfg, default_flow_style=False, sort_keys=False), encoding="utf-8")
print(f"Patched config: fp16={cfg['training']['fp16']}, batch_size={cfg['training']['batch_size']}")

## 1. Imports

In [ ]:
import json
import logging

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

from rag_filtering.config.loader import load_yaml, resolve_path
from verifier_system import VerifierSystem

logging.basicConfig(level=logging.INFO, format="%(levelname)s:%(name)s:%(message)s")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
print("Setup complete.")

## 2. Config preview

In [ ]:
cfg = load_yaml(CONFIG_PATH)
print("Model:", cfg["model"]["name"])
print("Checkpoint:", cfg["model"]["model_save_dir"])
print("Results:", cfg["evaluation"]["results_dir"])
print("Epochs:", cfg["training"]["num_epochs"], "| batch:", cfg["training"]["batch_size"], "| fp16:", cfg["training"]["fp16"])
print("\nReflection targets:")
print("  accept:", cfg["reflection_tokens"]["accept_target"])
print("  reject:", cfg["reflection_tokens"]["reject_target"])

## 3. Load data

In [ ]:
verifier = VerifierSystem(config_path=CONFIG_PATH)
verifier.load_data()

print(f"Train: {len(verifier.train_df)}, Val: {len(verifier.val_df)}, Test: {len(verifier.test_df)}")
print("Label balance (train):", verifier.train_df["label"].value_counts().to_dict())

## 4. Prompt and target preview

In [ ]:
sample_pos = verifier.train_df[verifier.train_df["label"] == 1].iloc[0]
sample_neg = verifier.train_df[verifier.train_df["label"] == 0].iloc[0]

for label, row in [(1, sample_pos), (0, sample_neg)]:
    ctx = verifier._extract_context(str(row["context"]))
    prompt = verifier._build_prompt(str(row["question"]), ctx, str(row["answer"]))
    target = verifier._build_target(int(row["label"]))
    print(f"\n=== label={label} ===")
    print("PROMPT:", prompt[:400], "..." if len(prompt) > 400 else "")
    print("TARGET:", target)

## 5. Train or load checkpoint

Default on Kaggle: `RUN_TRAIN = True`. Set `RUN_TRAIN = False` only if
`models/answer_verifier/` already exists in `/kaggle/working/ragas-evaluation`.

In [ ]:
checkpoint_dir = resolve_path(cfg["model"]["model_save_dir"])

if RUN_TRAIN:
    save_path = verifier.train(resume_from_checkpoint=RESUME_TRAINING)
    print(f"Training complete. Saved to {save_path}")
else:
    verifier.load_model()
    print(f"Loaded checkpoint from {checkpoint_dir}")

## 6. Evaluate

In [ ]:
metrics = verifier.evaluate(split=SPLIT)

summary = {
    "split": metrics["split"],
    "n_samples": metrics["n_samples"],
    "accuracy": metrics["accuracy"],
    "precision": metrics["precision"],
    "recall": metrics["recall"],
    "f1": metrics["f1"],
    "fpr": metrics["fpr"],
    "tp": metrics["tp"],
    "tn": metrics["tn"],
    "fp": metrics["fp"],
    "fn": metrics["fn"],
}
print(json.dumps(summary, indent=2))

results_path = resolve_path(cfg["evaluation"]["results_dir"]) / f"metrics_{SPLIT}.json"
print(f"\nMetrics saved to {results_path}")

In [ ]:
y_true = [0] * metrics["tn"] + [0] * metrics["fp"] + [1] * metrics["fn"] + [1] * metrics["tp"]
y_pred = [0] * metrics["tn"] + [1] * metrics["fp"] + [0] * metrics["fn"] + [1] * metrics["tp"]
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

fig, ax = plt.subplots()
ConfusionMatrixDisplay(
    cm,
    display_labels=["REJECT (0)", "ACCEPT (1)"],
).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Self-RAG Verifier — {SPLIT} split")
plt.tight_layout()
plt.show()

## 7. Sample predictions

In [ ]:
eval_df = verifier.test_df if SPLIT == "test" else (
    verifier.val_df if SPLIT == "val" else verifier.train_df
)

preview = eval_df.sample(n=min(5, len(eval_df)), random_state=cfg["data"]["seed"])
rows = []

for _, row in preview.iterrows():
    pred = verifier.predict(
        str(row["question"]),
        str(row["context"]),
        str(row["answer"]),
    )
    rows.append({
        "label": int(row["label"]),
        "decision": pred["decision"],
        "correct": int(row["label"]) == (1 if pred["accept"] else 0),
        "raw_output": pred["raw_output"],
        "question": str(row["question"])[:80],
        "answer": str(row["answer"])[:80],
    })

pd.DataFrame(rows)

## 8. Error spot-check

In [ ]:
batch_size = cfg["training"]["batch_size"]
predictions = verifier.predict_batch(
    questions=eval_df["question"].astype(str).tolist(),
    contexts=eval_df["context"].astype(str).tolist(),
    answers=eval_df["answer"].astype(str).tolist(),
    batch_size=batch_size,
)

eval_df = eval_df.copy()
eval_df["pred_accept"] = [p["accept"] for p in predictions]
eval_df["raw_output"] = [p["raw_output"] for p in predictions]

false_positives = eval_df[(eval_df["label"] == 0) & (eval_df["pred_accept"])].head(3)
false_negatives = eval_df[(eval_df["label"] == 1) & (~eval_df["pred_accept"])].head(3)

print(f"False positives: {len(eval_df[(eval_df['label'] == 0) & (eval_df['pred_accept'])])}")
print(f"False negatives: {len(eval_df[(eval_df['label'] == 1) & (~eval_df['pred_accept'])])}")

for title, subset in [("FALSE POSITIVE (accepted bad answer)", false_positives),
                      ("FALSE NEGATIVE (rejected good answer)", false_negatives)]:
    print(f"\n=== {title} ===")
    for _, row in subset.iterrows():
        print(f"Q: {str(row['question'])[:100]}")
        print(f"A: {str(row['answer'])[:120]}")
        print(f"Generated: {row['raw_output']}\n")

## 9. Package outputs for download

Zips are written to `/kaggle/working/` — download from the notebook **Output** tab
after **Save Version**.

In [ ]:
output_root = Path("/kaggle/working")
for rel in [cfg["model"]["model_save_dir"], cfg["evaluation"]["results_dir"]]:
    src = resolve_path(rel)
    if not src.exists():
        print(f"Skip (missing): {src}")
        continue
    zip_path = output_root / src.name
    if zip_path.with_suffix(".zip").exists():
        zip_path.with_suffix(".zip").unlink()
    archive = shutil.make_archive(str(zip_path), "zip", src)
    print(f"Created {archive}")

print("\nDone. Save this notebook version to persist outputs on Kaggle.")